# 04 — Why petrol drivers haven't converted

**Question:** what do the petrol-only drivers in the Lagos driver survey (S47) say stops them converting to CNG,
what would change their minds, and would they gain if they did convert?

**Data:** `data/processed/survey_clean.csv` (made by `src/clean_survey.py`). Method and limitations:
`docs/survey_method.md`.

**Which rows:** questions about barriers and willingness to convert use **all 12 petrol-only drivers**, because the
quality flags are about implausible *money* figures and don't affect these answers. Anything involving money
(sections 4 and 5) uses **unflagged rows only**, which leaves **7 petrol drivers**.

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.float_format = "{:,.2f}".format

survey = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "survey_clean.csv")
petrol = survey[~survey["uses_cng"]].copy()
print(f"All responses: {len(survey)} | petrol-only drivers: {len(petrol)}")

All responses: 25 | petrol-only drivers: 12


## 2. Barriers to conversion (all 12 petrol drivers)

The question was **tick all that apply**, so one driver can count towards several barriers and the percentages add
up to more than 100%. Each percentage is "share of the 12 drivers who ticked this".

In the saved CSV the `barrier_*` columns are blank for CNG users (they weren't asked). For petrol drivers they are
always True or False, so we can safely convert them to `bool` and add them up: `True` counts as 1, `False` as 0.

In [2]:
# Readable names for the chart and table (the answer text from the form, shortened).
BARRIER_LABELS = {
    "barrier_no_station": "No CNG station near where I work",
    "barrier_long_queues": "Queues at CNG stations are too long",
    "barrier_too_expensive": "Conversion is too expensive",
    "barrier_safety": "Worried about safety",
    "barrier_not_my_vehicle": "The vehicle is not mine",
}
barrier_cols = [c for c in petrol.columns if c.startswith("barrier_")]
assert set(barrier_cols) == set(BARRIER_LABELS), "a barrier column is missing a label"
assert petrol[barrier_cols].notna().all().all(), "petrol drivers should have no blank barrier answers"

ticked = petrol[barrier_cols].astype(bool)
barriers = pd.DataFrame({
    "barrier": [BARRIER_LABELS[c] for c in barrier_cols],
    "drivers": ticked.sum().values,
})
barriers["percent"] = barriers["drivers"] / len(petrol)
barriers = barriers.sort_values("drivers", ascending=False, kind="stable").reset_index(drop=True)

print(f"Barriers ticked per driver: median {ticked.sum(axis=1).median():g}, "
      f"range {ticked.sum(axis=1).min()}-{ticked.sum(axis=1).max()}")
barriers.style.format({"percent": "{:.0%}"}).hide(axis="index")

Barriers ticked per driver: median 2.5, range 1-4


barrier,drivers,percent
No CNG station near where I work,10,83%
Queues at CNG stations are too long,9,75%
Conversion is too expensive,7,58%
Worried about safety,1,8%
The vehicle is not mine,1,8%


### Chart

Design choices: a single series, so one colour and no legend (the title names what the bars are). Each bar is
labelled with its count and percentage, so the chart can be read without the axis. The axis runs to 12 so the bars
read as "out of 12". Plotly draws horizontal bars from the bottom up, so we reverse the order to put the most common
barrier at the top.

In [3]:
BAR = "#2a78d6"
INK = "#0b0b0b"
INK_MUTED = "#52514e"

plot_data = barriers.iloc[::-1]   # reversed: plotly's first bar is at the bottom
fig = go.Figure(go.Bar(
    x=plot_data["drivers"], y=plot_data["barrier"], orientation="h",
    marker=dict(color=BAR, cornerradius=4),
    width=0.6,
    text=[f"{d}  ({p:.0%})" for d, p in zip(plot_data["drivers"], plot_data["percent"])],
    textposition="outside", textfont=dict(color=INK, size=13), cliponaxis=False,
    hovertemplate="<b>%{y}</b><br>%{x} of " + str(len(petrol)) + " drivers<extra></extra>",
))
fig.update_layout(
    title=dict(
        text="What stops petrol drivers converting to CNG?<br>"
             f"<sup>Lagos driver survey (S47), {len(petrol)} petrol-only drivers. "
             "Tick all that apply, so drivers can appear in several bars.</sup>",
        x=0, xanchor="left",
    ),
    template="plotly_white",
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK),
    xaxis=dict(title=f"Number of drivers (of {len(petrol)})", range=[0, len(petrol)], dtick=2,
               gridcolor="#ecebe7", zeroline=False),
    yaxis=dict(title=None, ticksuffix="  "),
    hoverlabel=dict(bgcolor="white", font=dict(color=INK)),
    showlegend=False, width=820, height=420, margin=dict(t=90, l=20, r=60, b=60),
)
fig.show()

In [4]:
out_path = FIG_DIR / "barriers.html"
fig.write_html(out_path, include_plotlyjs="cdn")   # cdn: small file, needs internet to render (see notebook 01)
print(f"Saved {out_path.relative_to(PROJECT_ROOT)}")

Saved outputs\figures\barriers.html


## 3. What would change their minds (all 12 petrol drivers)

Two questions: would you convert if there was a CNG station within 10 minutes of where you work
(`convert_if_station`), and would you convert if the conversion was free (`convert_if_free`)?

In [5]:
answers = pd.DataFrame({
    "convert_if_station": petrol["convert_if_station"].value_counts(),
    "convert_if_free": petrol["convert_if_free"].value_counts(),
}).reindex(["Yes", "Not sure", "No"]).fillna(0).astype(int)
answers

,convert_if_station,convert_if_free
Yes,10,11
Not sure,1,0
No,1,1


In [6]:
# pd.crosstab counts every combination of the two answers; margins=True adds the totals.
pd.crosstab(petrol["convert_if_station"], petrol["convert_if_free"],
            rownames=["If a station were nearby"], colnames=["If conversion were free"], margins=True,
            margins_name="Total")

If conversion were free,No,Yes,Total
If a station were nearby,,,
No,1,0,1
Not sure,0,1,1
Yes,0,10,10
Total,1,11,12


The same answers grouped by **how many conditions would get a "Yes"**. "Not sure" is counted as not a yes.

In [7]:
station_yes = petrol["convert_if_station"] == "Yes"
free_yes = petrol["convert_if_free"] == "Yes"

summary = pd.Series({
    "Yes to both": (station_yes & free_yes).sum(),
    "Yes to a nearby station only": (station_yes & ~free_yes).sum(),
    "Yes to free conversion only": (~station_yes & free_yes).sum(),
    "Yes to neither": (~station_yes & ~free_yes).sum(),
}, name="drivers")
assert summary.sum() == len(petrol)
summary.to_frame()

,drivers
Yes to both,10
Yes to a nearby station only,0
Yes to free conversion only,1
Yes to neither,1


## 4. Petrol vs CNG drivers (unflagged rows only)

Money figures, so flagged rows are dropped. **Only 7 of the 12 petrol drivers are unflagged** (all five flagged
responses are petrol drivers), so the petrol column rests on 7 people.

`gross_per_hour` is the median of each driver's own gross ÷ hours. It is not the median gross divided by the median
hours, which would give a different number.

In [8]:
unflagged = survey[survey["quality_flag"].isna()]
group = unflagged["uses_cng"].map({False: "Petrol only", True: "CNG (incl. bi-fuel)"})
compare_cols = ["gross_per_day", "hours_per_day", "gross_per_hour", "days_per_week"]

comparison = unflagged.groupby(group)[compare_cols].median().T
comparison.loc["n"] = group.value_counts()
comparison = comparison[["Petrol only", "CNG (incl. bi-fuel)"]]
comparison.style.format("{:,.0f}")

uses_cng,Petrol only,CNG (incl. bi-fuel)
gross_per_day,"120,000","120,000"
hours_per_day,12,9
gross_per_hour,"10,000","11,538"
days_per_week,7,7
n,7,13


## 5. Estimated gain if the unflagged petrol drivers converted for free

> ⚠️ **Assumption:** these drivers have never used CNG, so we have no CNG figures for them. We apply the **median
> CNG user's** fuel saving % and daily queue hours from the same survey to each petrol driver. That assumes they
> would save and queue like the median CNG user, whatever their vehicle, route or location.

For each unflagged petrol driver:

| Quantity | Formula |
|---|---|
| Fuel saving | petrol spend per day × median CNG user fuel saving % |
| Earnings per hour | (gross per day − petrol spend per day) ÷ hours per day |
| Queue cost | median CNG user daily queue hours × earnings per hour |
| Net benefit | fuel saving − queue cost (free conversion, so no loan cost) |

As in notebook 03, "earnings per hour" is gross minus fuel, before any owner remittance or app commission.

In [9]:
cng_users = unflagged[unflagged["uses_cng"]]
median_saving_pct = cng_users["fuel_saving_pct"].median()
median_daily_queue = cng_users["daily_queue_hours"].median()
print(f"Applied from {len(cng_users)} unflagged CNG users: fuel saving {median_saving_pct:.1%}, "
      f"daily queue {median_daily_queue:.2f} hours")

gain = unflagged[~unflagged["uses_cng"]].copy()
gain["fuel_saving"] = gain["petrol_spend_per_day"] * median_saving_pct
gain["earnings_per_hour"] = (gain["gross_per_day"] - gain["petrol_spend_per_day"]) / gain["hours_per_day"]
gain["queue_cost"] = median_daily_queue * gain["earnings_per_hour"]
gain["net_benefit"] = gain["fuel_saving"] - gain["queue_cost"]

gain_cols = ["vehicle_type", "gross_per_day", "hours_per_day", "petrol_spend_per_day", "fuel_saving",
             "earnings_per_hour", "queue_cost", "net_benefit", "convert_if_station", "convert_if_free"]
gain.set_index("respondent_id")[gain_cols].style.format(
    {c: "{:,.0f}" for c in gain_cols if c not in ("vehicle_type", "convert_if_station", "convert_if_free")}
)

Applied from 13 unflagged CNG users: fuel saving 81.7%, daily queue 2.25 hours


,vehicle_type,gross_per_day,hours_per_day,petrol_spend_per_day,fuel_saving,earnings_per_hour,queue_cost,net_benefit,convert_if_station,convert_if_free
respondent_id,,,,,,,,,,
R16,keke,"80,000",12,"9,000","7,357","5,917","13,312","-5,956",Not sure,Yes
R18,taxi,"130,000",9,"25,000","20,435","11,667","26,250","-5,815",Yes,Yes
R19,danfo,"145,000",11,"35,000","28,609","10,000","22,500","6,109",Yes,Yes
R20,danfo,"80,000",13,"28,000","22,887","4,000","9,000","13,887",Yes,Yes
R21,danfo,"50,000",6,"15,000","12,261","5,833","13,125",-864,Yes,Yes
R23,ride_hailing,"120,000",12,"35,000","28,609","7,083","15,938","12,671",Yes,Yes
R24,keke,"140,000",13,"28,000","22,887","8,615","19,385","3,502",Yes,Yes


In [10]:
n_gain = (gain["net_benefit"] > 0).sum()
print(f"Would gain (net benefit > 0): {n_gain} of {len(gain)} unflagged petrol drivers")
print(f"Median net benefit: NGN {gain['net_benefit'].median():,.0f} per day")

Would gain (net benefit > 0): 4 of 7 unflagged petrol drivers
Median net benefit: NGN 3,502 per day


## 6. What the results mean

**Access is the main barrier, not price.** Of the 12 petrol drivers, 10 say there is no CNG station near where they
work and 9 say the queues are too long. 7 say conversion is too expensive. Safety and not owning the vehicle were
ticked by one driver each. Most drivers ticked more than one barrier, often all three of station access, queues and
cost.

**Nearly all of them say they would convert if things changed.** 10 of the 12 would convert under either condition: a
station within 10 minutes, or a free conversion. One more would convert if it were free but is not sure about a
nearby station. One driver would convert under neither. That driver had ticked all four of cost, access, queues and
safety, and is the only one worried about safety. Because nearly everyone says yes to both, the survey can't tell
which lever matters more. It shows that stated interest is high, not which policy would work best.

**Petrol and CNG drivers earn similar amounts per day.** Among unflagged drivers, the median petrol driver grosses
₦120,000 a day, the same as the median CNG driver. Petrol drivers report longer days (12 hours against 9), so they earn
a little less per hour (₦10,000 against about ₦11,500). Both groups' median is 7 days a week. With only 7 petrol
drivers in this comparison, these differences could easily be chance.

**Converting would not pay for everyone.** If the 7 unflagged petrol drivers converted for free, and saved and queued
like the median CNG user (81.7% fuel saving, 2.25 hours of queuing a day), 4 of the 7 would come out ahead. The other
3 would lose a little: two because they spend relatively little on petrol (R16, a keke, ₦9,000 a day; R21, a danfo,
₦15,000), and one (R18, a taxi) because queue time costs a lot at about ₦11,700 an hour. This suggests the
queues that drivers name as a barrier are a real cost, not just an inconvenience. For some drivers, free conversion
alone would not make CNG pay unless queues were shorter too.

**Limitations:**
- **12 drivers.** The barrier and willingness results come from 12 people; the money comparisons from 7. Percentages
  move by about 8 points with each driver. The results describe these drivers, not Lagos petrol drivers in general.
- **Recruitment.** Drivers were approached in person at CNG stations and car parks in Lagos
  (`docs/survey_method.md`). Petrol drivers found at those places may differ from others, for example by being
  closer to or further from stations than average.
- **Tick all that apply.** Drivers could tick as many barriers as they liked and weren't asked which mattered most.
  The counts show how often a barrier is mentioned, not how much it weighs in the decision.
- **Stated intentions.** "Would you convert if…" answers are hypothetical, and people often say yes to a free offer
  they would not take up in practice.
- **Gain estimate is an assumption.** It applies CNG users' median fuel saving and queue time to drivers who have
  never used CNG. The CNG medians come mostly from ride-hailing and danfo drivers (only one keke), and rest on recalled
  petrol spend (notebook 03). "Earnings per hour" is gross minus fuel, before owner remittance or commission.